In [1]:
from dotenv import load_dotenv
load_dotenv("/Users/thomasmyles/dev/betting/.env")
print("env loaded")

env loaded


In [2]:
import os
import json
import time
import requests
from pathlib import Path

API_KEY   = os.environ["ODDS_API_KEY"]
BASE_URL  = "https://api.the-odds-api.com/v4"
SPORT     = "americanfootball_nfl"
REGIONS   = "us,us2"   # required param for historical endpoint
OUT_DIR   = Path("data/nfl/sb_lx_markets")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Pre-game closing snapshot: Feb 8 2026 6:00 PM ET = 23:00 UTC
# Game kicked off at 23:30 UTC (6:30 PM ET)
SNAPSHOT = "2026-02-08T23:00:00Z"

def remaining(resp):
    return resp.headers.get('x-requests-remaining', '?')

print(f"Output dir: {OUT_DIR.resolve()}")
print(f"Snapshot  : {SNAPSHOT}")

Output dir: /Users/thomasmyles/dev/betting/notebooks/data/nfl/sb_lx_markets
Snapshot  : 2026-02-08T23:00:00Z


## Step 1 — Find the Super Bowl LX event ID

In [3]:
# h2h only to find the event at low cost
resp = requests.get(
    f"{BASE_URL}/historical/sports/{SPORT}/odds",
    params={
        "apiKey":     API_KEY,
        "markets":    "h2h",
        "regions":    REGIONS,
        "oddsFormat": "american",
        "dateFormat": "iso",
        "date":       SNAPSHOT,
    },
    timeout=30,
)
print(f"Status: {resp.status_code}  |  remaining: {remaining(resp)}")

events = resp.json().get("data", [])
print(f"Events in snapshot: {len(events)}")
for e in events:
    print(f"  {e['id']}  |  {e['home_team']} vs {e['away_team']}  |  {e['commence_time']}")

Status: 200  |  remaining: 96460
Events in snapshot: 1
  b64e3587d7a4cf01a568e7150a2a1aec  |  New England Patriots vs Seattle Seahawks  |  2026-02-08T23:30:00Z


In [4]:
sb = next(
    (e for e in events
     if "Seattle" in (e["home_team"] + e["away_team"])
     or "New England" in (e["home_team"] + e["away_team"])),
    None
)
assert sb, "Super Bowl event not found — check snapshot or team names above"

EVENT_ID = sb["id"]
print(f"Super Bowl LX")
print(f"  Home : {sb['home_team']}")
print(f"  Away : {sb['away_team']}")
print(f"  Start: {sb['commence_time']}")
print(f"  ID   : {EVENT_ID}")

Super Bowl LX
  Home : New England Patriots
  Away : Seattle Seahawks
  Start: 2026-02-08T23:30:00Z
  ID   : b64e3587d7a4cf01a568e7150a2a1aec


## Step 2 — All known NFL markets

In [5]:
ALL_NFL_MARKETS = [
    # game markets
    "h2h", "spreads", "totals",
    "alternate_spreads", "alternate_totals", "team_totals", "alternate_team_totals",
    # quarters
    "h2h_q1", "h2h_q2", "h2h_q3", "h2h_q4",
    "h2h_3_way_q1", "h2h_3_way_q2", "h2h_3_way_q3", "h2h_3_way_q4",
    "spreads_q1", "spreads_q2", "spreads_q3", "spreads_q4",
    "alternate_spreads_q1", "alternate_spreads_q2", "alternate_spreads_q3", "alternate_spreads_q4",
    "totals_q1", "totals_q2", "totals_q3", "totals_q4",
    "alternate_totals_q1", "alternate_totals_q2", "alternate_totals_q3", "alternate_totals_q4",
    "team_totals_q1", "team_totals_q2", "team_totals_q3", "team_totals_q4",
    "alternate_team_totals_q1", "alternate_team_totals_q2", "alternate_team_totals_q3", "alternate_team_totals_q4",
    # halves
    "h2h_h1", "h2h_h2", "h2h_3_way_h1", "h2h_3_way_h2",
    "spreads_h1", "spreads_h2", "alternate_spreads_h1", "alternate_spreads_h2",
    "totals_h1", "totals_h2", "alternate_totals_h1", "alternate_totals_h2",
    "team_totals_h1", "team_totals_h2", "alternate_team_totals_h1", "alternate_team_totals_h2",
    # passing
    "player_pass_tds", "player_pass_yds", "player_pass_yds_q1",
    "player_pass_attempts", "player_pass_completions", "player_pass_interceptions",
    "player_pass_longest_completion",
    # rushing
    "player_rush_tds", "player_rush_yds", "player_rush_attempts", "player_rush_longest",
    # receiving
    "player_receptions", "player_reception_tds", "player_reception_yds", "player_reception_longest",
    # defense
    "player_tackles_assists", "player_solo_tackles", "player_sacks", "player_defensive_interceptions",
    # kicking
    "player_field_goals", "player_kicking_points", "player_pats",
    # combos
    "player_pass_rush_yds", "player_rush_reception_yds",
    "player_pass_rush_reception_yds", "player_rush_reception_tds", "player_pass_rush_reception_tds",
    # scoring specials
    "player_tds_over", "player_1st_td", "player_anytime_td", "player_last_td",
    # alternates
    "player_pass_tds_alternate", "player_pass_yds_alternate",
    "player_pass_attempts_alternate", "player_pass_completions_alternate",
    "player_pass_interceptions_alternate", "player_pass_longest_completion_alternate",
    "player_rush_tds_alternate", "player_rush_yds_alternate",
    "player_rush_attempts_alternate", "player_rush_longest_alternate",
    "player_reception_tds_alternate", "player_reception_yds_alternate",
    "player_receptions_alternate", "player_reception_longest_alternate",
    "player_tackles_assists_alternate", "player_solo_tackles_alternate",
    "player_sacks_alternate", "player_field_goals_alternate",
    "player_kicking_points_alternate", "player_pats_alternate",
    "player_pass_rush_yds_alternate", "player_rush_reception_yds_alternate",
    "player_pass_rush_reception_yds_alternate", "player_rush_reception_tds_alternate",
    "player_pass_rush_reception_tds_alternate",
]

print(f"Total markets to probe: {len(ALL_NFL_MARKETS)}")

Total markets to probe: 111


## Step 3 — Probe which markets are available for this event

4 markets per call. Collect keys that return at least one bookmaker with data.

In [6]:
def fetch_event_markets(event_id, markets, snapshot, delay=0.2):
    resp = requests.get(
        f"{BASE_URL}/historical/sports/{SPORT}/events/{event_id}/odds",
        params={
            "apiKey":     API_KEY,
            "markets":    ",".join(markets),
            "regions":    REGIONS,
            "oddsFormat": "american",
            "dateFormat": "iso",
            "date":       snapshot,
        },
        timeout=30,
    )
    time.sleep(delay)
    return resp


def batched(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


BATCH_SIZE = 4
available_markets = []

print(f"Probing {len(ALL_NFL_MARKETS)} markets in batches of {BATCH_SIZE}...\n")

for mkt_batch in batched(ALL_NFL_MARKETS, BATCH_SIZE):
    resp = fetch_event_markets(EVENT_ID, mkt_batch, SNAPSHOT)
    label = ",".join(mkt_batch[:2]) + (",..." if len(mkt_batch) > 2 else "")
    print(f"  [{label}] → {resp.status_code}  remaining={remaining(resp)}")

    if resp.status_code != 200:
        print(f"    ERROR: {resp.text[:200]}")
        continue

    event_data = resp.json().get("data", {})
    if not event_data:
        continue

    found_keys = set()
    for bookmaker in event_data.get("bookmakers", []):
        for mkt in bookmaker.get("markets", []):
            found_keys.add(mkt["key"])

    for k in mkt_batch:
        if k in found_keys:
            available_markets.append(k)

print(f"\nAvailable markets ({len(available_markets)}):")
for m in available_markets:
    print(f"  {m}")

Probing 111 markets in batches of 4...

  [h2h,spreads,...] → 200  remaining=96380
  [alternate_totals,team_totals,...] → 200  remaining=96300
  [h2h_q2,h2h_q3,...] → 200  remaining=96220
  [h2h_3_way_q2,h2h_3_way_q3,...] → 200  remaining=96140
  [spreads_q2,spreads_q3,...] → 200  remaining=96060
  [alternate_spreads_q2,alternate_spreads_q3,...] → 200  remaining=96040
  [totals_q2,totals_q3,...] → 200  remaining=95960
  [alternate_totals_q2,alternate_totals_q3,...] → 200  remaining=95940
  [team_totals_q2,team_totals_q3,...] → 200  remaining=95860
  [alternate_team_totals_q2,alternate_team_totals_q3,...] → 200  remaining=95820
  [h2h_h2,h2h_3_way_h1,...] → 200  remaining=95740
  [spreads_h2,alternate_spreads_h1,...] → 200  remaining=95660
  [totals_h2,alternate_totals_h1,...] → 200  remaining=95580
  [team_totals_h2,alternate_team_totals_h1,...] → 200  remaining=95500
  [player_pass_yds,player_pass_yds_q1,...] → 200  remaining=95420
  [player_pass_interceptions,player_pass_longest_comp

## Step 4 — Full fetch: one call per market, save raw JSON

In [7]:
fetched = {}

for i, market in enumerate(available_markets, 1):
    out_path = OUT_DIR / f"{market}.json"

    if out_path.exists():
        print(f"  [{i}/{len(available_markets)}] {market} — cached")
        with open(out_path) as f:
            fetched[market] = json.load(f)
        continue

    resp = fetch_event_markets(EVENT_ID, [market], SNAPSHOT)
    print(f"  [{i}/{len(available_markets)}] {market} → {resp.status_code}  remaining={remaining(resp)}")

    if resp.status_code != 200:
        print(f"    ERROR: {resp.text[:200]}")
        continue

    payload = resp.json()
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2)

    fetched[market] = payload

print(f"\nDone. {len(fetched)} markets saved to {OUT_DIR.resolve()}")

  [1/92] h2h → 200  remaining=94600
  [2/92] spreads → 200  remaining=94580
  [3/92] totals → 200  remaining=94560
  [4/92] alternate_spreads → 200  remaining=94540
  [5/92] alternate_totals → 200  remaining=94520
  [6/92] team_totals → 200  remaining=94500
  [7/92] alternate_team_totals → 200  remaining=94480
  [8/92] h2h_q1 → 200  remaining=94460
  [9/92] h2h_q2 → 200  remaining=94440
  [10/92] h2h_q3 → 200  remaining=94420
  [11/92] h2h_q4 → 200  remaining=94400
  [12/92] h2h_3_way_q1 → 200  remaining=94380
  [13/92] h2h_3_way_q2 → 200  remaining=94360
  [14/92] h2h_3_way_q3 → 200  remaining=94340
  [15/92] h2h_3_way_q4 → 200  remaining=94320
  [16/92] spreads_q1 → 200  remaining=94300
  [17/92] spreads_q2 → 200  remaining=94280
  [18/92] spreads_q3 → 200  remaining=94260
  [19/92] spreads_q4 → 200  remaining=94240
  [20/92] alternate_spreads_q1 → 200  remaining=94220
  [21/92] totals_q1 → 200  remaining=94200
  [22/92] totals_q2 → 200  remaining=94180
  [23/92] totals_q3 → 200  rem

## Step 5 — Flatten to DataFrame + save parquet

In [8]:
import pandas as pd

rows = []

for market, payload in fetched.items():
    event_data = payload.get("data", {})
    if not event_data:
        continue

    home_team   = event_data.get("home_team", "")
    away_team   = event_data.get("away_team", "")
    commence_ts = event_data.get("commence_time", "")
    snapshot_ts = payload.get("timestamp", SNAPSHOT)

    for bookmaker in event_data.get("bookmakers", []):
        book_key    = bookmaker["key"]
        book_title  = bookmaker["title"]
        last_update = bookmaker.get("last_update", "")

        for mkt in bookmaker.get("markets", []):
            mkt_key = mkt["key"]
            mkt_last_update = mkt.get("last_update", "")

            for outcome in mkt.get("outcomes", []):
                rows.append({
                    "event_id":         event_data.get("id", ""),
                    "home_team":        home_team,
                    "away_team":        away_team,
                    "commence_time":    commence_ts,
                    "snapshot_time":    snapshot_ts,
                    "market":           mkt_key,
                    "bookmaker":        book_key,
                    "bookmaker_title":  book_title,
                    "bookmaker_update": last_update,
                    "market_update":    mkt_last_update,
                    "outcome_name":     outcome.get("name", ""),
                    "outcome_desc":     outcome.get("description", ""),  # player name for props
                    "price":            outcome.get("price"),
                    "point":            outcome.get("point"),
                    "sid":              outcome.get("sid", ""),
                })

df = pd.DataFrame(rows)
print(f"Rows     : {len(df):,}")
print(f"Markets  : {df['market'].nunique()}")
print(f"Bookmakers: {df['bookmaker'].nunique()}")
print()
print(df.groupby('market')[['bookmaker']].nunique().rename(columns={'bookmaker': 'n_books'}).sort_values('n_books', ascending=False).to_string())

Rows     : 10,751
Markets  : 92
Bookmakers: 16

                                          n_books
market                                           
spreads                                        16
h2h                                            16
totals                                         14
totals_h1                                      12
h2h_q1                                         12
spreads_h1                                     12
player_reception_yds                           11
player_rush_yds                                11
player_pass_tds                                11
player_anytime_td                              11
player_tds_over                                11
h2h_h1                                         11
alternate_totals                               11
player_receptions                              11
spreads_q1                                     11
totals_q1                                      11
player_pass_yds                                11
pl

In [9]:
out_parquet = OUT_DIR / "sb_lx_all_markets.parquet"
df.to_parquet(out_parquet, index=False)
print(f"Saved: {out_parquet.resolve()}")
df.head()

Saved: /Users/thomasmyles/dev/betting/notebooks/data/nfl/sb_lx_markets/sb_lx_all_markets.parquet


,event_id,home_team,away_team,commence_time,snapshot_time,market,bookmaker,bookmaker_title,bookmaker_update,market_update,outcome_name,outcome_desc,price,point,sid
0,b64e3587d7a4cf01a568e7150a2a1aec,New England Patriots,Seattle Seahawks,2026-02-08T23:30:00Z,2026-02-08T22:55:37Z,h2h,draftkings,DraftKings,2026-02-08T22:54:21Z,2026-02-08T22:54:21Z,New England Patriots,,195,NaN,
1,b64e3587d7a4cf01a568e7150a2a1aec,New England Patriots,Seattle Seahawks,2026-02-08T23:30:00Z,2026-02-08T22:55:37Z,h2h,draftkings,DraftKings,2026-02-08T22:54:21Z,2026-02-08T22:54:21Z,Seattle Seahawks,,-238,NaN,
2,b64e3587d7a4cf01a568e7150a2a1aec,New England Patriots,Seattle Seahawks,2026-02-08T23:30:00Z,2026-02-08T22:55:37Z,h2h,lowvig,LowVig.ag,2026-02-08T22:54:37Z,2026-02-08T22:54:37Z,New England Patriots,,190,NaN,
3,b64e3587d7a4cf01a568e7150a2a1aec,New England Patriots,Seattle Seahawks,2026-02-08T23:30:00Z,2026-02-08T22:55:37Z,h2h,lowvig,LowVig.ag,2026-02-08T22:54:37Z,2026-02-08T22:54:37Z,Seattle Seahawks,,-227,NaN,
4,b64e3587d7a4cf01a568e7150a2a1aec,New England Patriots,Seattle Seahawks,2026-02-08T23:30:00Z,2026-02-08T22:55:37Z,h2h,espnbet,theScore Bet,2026-02-08T22:55:30Z,2026-02-08T22:55:30Z,New England Patriots,,202,NaN,
